#### Destilación Cuantizada (Quantized Distillation)

In [ ]:
import tensorflow as tf
from qkeras import QDense, QActivation

# 1. Cargar el Profesor (Tu modelo base congelado)
teacher_model = tf.keras.models.load_model('ruta_a_tu_modelo_etapa3.h5', custom_objects={'focal_loss': tu_focal_loss})
teacher_model.trainable = False

# 2. Definir el Estudiante (Mucho más pequeño y ya cuantizado a 8-bits)
student_model = tf.keras.Sequential([
    QDense(32, input_shape=(64,), kernel_quantizer="quantized_bits(8,0,alpha=1)", name='qdense_1'),
    tf.keras.layers.BatchNormalization(),
    QActivation("quantized_relu(4,0)"),
    QDense(16, kernel_quantizer="quantized_bits(8,0,alpha=1)", name='qdense_2'),
    tf.keras.layers.BatchNormalization(),
    QActivation("quantized_relu(4,0)"),
    QDense(1, activation='sigmoid', kernel_quantizer="quantized_bits(8,0,alpha=1)", name='output')
])

# 3. Clase personalizada de Destilación
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher):
        super(Distiller, self).__init__()
        self.teacher = teacher
        self.student = student

    def compile(self, optimizer, metrics, student_loss_fn, distillation_loss_fn, alpha=0.1, temperature=3):
        super(Distiller, self).compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn # Tu focal loss
        self.distillation_loss_fn = distillation_loss_fn # Usualmente Kullback-Leibler o MSE
        self.alpha = alpha
        self.temperature = temperature

    def train_step(self, data):
        x, y = data

        # Pasar los datos por el profesor (sin gradientes)
        teacher_predictions = self.teacher(x, training=False)

        with tf.GradientTape() as tape:
            # Pasar los datos por el estudiante
            student_predictions = self.student(x, training=True)

            # 1. Pérdida del estudiante contra las etiquetas reales (Focal Loss)
            student_loss = self.student_loss_fn(y, student_predictions)

            # 2. Pérdida de destilación (Estudiante intentando imitar al Profesor)
            # Como es clasificación binaria, MSE entre las salidas sigmoidales funciona muy bien.
            distillation_loss = self.distillation_loss_fn(teacher_predictions, student_predictions)

            # 3. Pérdida combinada ponderada por alpha
            loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

        # Calcular y aplicar gradientes al ESTUDIANTE
        trainable_vars = self.student.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        # Actualizar métricas
        self.compiled_metrics.update_state(y, student_predictions)
        return {m.name: m.result() for m in self.metrics}

    def call(self, x):
        return self.student(x)

# 4. Instanciar, compilar y entrenar
distiller = Distiller(student=student_model, teacher=teacher_model)
distiller.compile(
    optimizer=tf.keras.optimizers.Adam(),
    metrics=[tf.keras.metrics.AUC()],
    student_loss_fn=tu_focal_loss,
    distillation_loss_fn=tf.keras.losses.MeanSquaredError(),
    alpha=0.5 # Balance entre imitar al profesor y acertar las etiquetas reales
)

distiller.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

# 5. Guardar el modelo estudiante final
student_model.save('modelo_estudiante_qat.h5')